## 1. Import et Fonctions Utilitaires

In [1]:
using DataFrames, Printf, BenchmarkTools, Plots

SyntaxError: invalid syntax (4181836154.py, line 1)

## 2. Fonction de Lecture d'Instances

In [2]:
function readKnaptxtInstance(filename)
    price=[]
    weight=[]
    KnapCap=[]
    open(filename) do f
        for i in 1:3
            tok = split(readline(f))
            if(tok[1] == "ListPrices=")
                for i in 2:(length(tok)-1)
                    push!(price,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "ListWeights=")
                for i in 2:(length(tok)-1)
                    push!(weight,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "Capacity=")
                push!(KnapCap, parse(Int64, tok[2]))
            else
                println("Unknown read :", tok)
            end 
        end
    end
    capacity=KnapCap[1]
    return price, weight, capacity
end

# Test
prices, weights, capacity = readKnaptxtInstance("InstancesKnapSack/test.opb.txt")
println("✓ Fonction de lecture OK")
println("  Exemple : prix=$prices, poids=$weights, capacité=$capacity")

SyntaxError: invalid syntax (4102875632.py, line 1)

## 3. TP3 - Programmation Dynamique

In [3]:
function solveKnapDP(filename)
    ## Loading data ##
    prices, weights, capacity_bag = readKnaptxtInstance(filename)

    ## Initialisation of the matrix ##
    n_items = length(prices)
    n_columns = capacity_bag + 1
    matrix_best_price = zeros(n_items, n_columns)

    ## Fulfilling th first row ##
    for capacity in weights[1]+1:n_columns
        matrix_best_price[1,capacity] = prices[1]
    end
    
    ## Fulfilling the matrix with the recursive formula ##
    for item in 2:n_items
        for capacity in 1:n_columns
            if capacity >= weights[item]+1
                matrix_best_price[item, capacity] = max(matrix_best_price[item-1, capacity], matrix_best_price[item-1, capacity-weights[item]] + prices[item])
            else
                matrix_best_price[item, capacity] = matrix_best_price[item-1, capacity]
            end
        end
    end

    best_cost = matrix_best_price[n_items, n_columns]

    ## Backtracking to identify items selected
    remaining_capacity = capacity_bag + 1 
    list_items_selected = []

    ## Backtracking rows 2:n_items
    for item in n_items:-1:2
        if matrix_best_price[item, remaining_capacity] != matrix_best_price[item-1, remaining_capacity]
            push!(list_items_selected, item)
            remaining_capacity -= weights[item]
        end
    end

    ## Checking whether item n°1 has been selected 
    if matrix_best_price[1,remaining_capacity] > 0
        push!(list_items_selected, 1)
    end

    return best_cost, reverse(list_items_selected)
end

println("✓ Fonction TP3 (PD) OK")

SyntaxError: invalid syntax (2655358380.py, line 1)

## 4. Instances à Tester

In [ ]:
instances = [
    ("test.opb.txt", "InstancesKnapSack/test.opb.txt"),
    ("Similar_Weights_100", "InstancesKnapSack/Similar_Weights/KnapSack_100_1000_-995.opb.txt"),
    ("Strongly_Correlated_100", "InstancesKnapSack/Strongly_Correlated/KnapSack_100_1000_-2397.opb.txt"),
    ("Uncorrelated_100", "InstancesKnapSack/Uncorrelated/KnapSack_100_1000_-9147.opb.txt"),
    ("Weakly_Correlated_100", "InstancesKnapSack/Weakly_Correlated/KnapSack_100_1000_-1514.opb.txt"),
]

println("$(length(instances)) instances configurées")

## 5. Exécution TP3 et Collecte des Résultats

In [ ]:
mutable struct Resultat
    name::String
    filepath::String
    tp3_cost::Union{Float64, Missing}
    tp3_items::Union{Vector, Missing}
    tp3_time::Union{Float64, Missing}
    tp2_cost::Union{Float64, Missing}
    tp2_items::Union{Vector, Missing}
    tp2_time::Union{Float64, Missing}
end

resultats = Resultat[]

println("="^80)
println(" EXÉCUTION TP3 - PROGRAMMATION DYNAMIQUE")
println("="^80)
println()

for (name, filepath) in instances
    if !isfile(filepath)
        println("❌ Fichier non trouvé : $filepath")
        push!(resultats, Resultat(name, filepath, missing, missing, missing, missing, missing, missing))
        continue
    end
    
    println("📊 Instance : $name")
    
    try
        tp3_time = @elapsed begin
            tp3_cost, tp3_items = solveKnapDP(filepath)
        end
        
        println("   ✓ Coût = $(Int(tp3_cost)), Objets = $tp3_items")
        println("   ⏱️  Temps = $(round(tp3_time*1000, digits=3)) ms")
        
        push!(resultats, Resultat(name, filepath, tp3_cost, tp3_items, tp3_time, missing, missing, missing))
        
    catch e
        println("   ❌ ERREUR : $e")
        push!(resultats, Resultat(name, filepath, missing, missing, missing, missing, missing, missing))
    end
    println()
end

println("="^80)

## 6. Tableau Récapitulatif TP3

In [ ]:
# Créer un DataFrame pour affichage
df_tp3 = DataFrame(
    Instance = [r.name for r in resultats],
    TP3_Coût = [ismissing(r.tp3_cost) ? "NA" : Int(r.tp3_cost) for r in resultats],
    TP3_Temps_ms = [ismissing(r.tp3_time) ? "NA" : round(r.tp3_time*1000, digits=3) for r in resultats],
)

println("\n📋 RÉSULTATS TP3 (Programmation Dynamique)\n")
println(df_tp3)

## 7. Comparaison TP2 vs TP3 (À COMPLÉTER)

### Instructions pour intégrer les résultats TP2 :

1. **Exécutez d'abord TOUTES les cellules du NotebookTP2.ipynb**
2. **Récupérez les résultats** (coût et temps) pour chaque instance
3. **Complétez les cellules ci-dessous** avec vos résultats TP2

In [ ]:
# À COMPLÉTER : Ajoutez les résultats TP2 ici

# Exemple format :
# resultats[1].tp2_cost = 65     # Coût optimal pour test.opb.txt
# resultats[1].tp2_time = 0.005  # Temps en secondes
# resultats[1].tp2_items = [2, 4]

# À faire pour chaque instance...

println("À compléter avec les résultats du TP2")

## 8. Tableau Comparatif Complet

In [ ]:
# DataFrame comparatif
df_comparison = DataFrame(
    Instance = [r.name for r in resultats],
    TP2_Coût = [ismissing(r.tp2_cost) ? "À compl." : Int(r.tp2_cost) for r in resultats],
    TP3_Coût = [ismissing(r.tp3_cost) ? "NA" : Int(r.tp3_cost) for r in resultats],
    Écart = [
        if ismissing(r.tp2_cost) || ismissing(r.tp3_cost)
            "À compl."
        elseif r.tp2_cost ≈ r.tp3_cost
            "✓ 0"
        else
            "✗ $(abs(r.tp2_cost - r.tp3_cost))"
        end
        for r in resultats
    ],
    TP2_Temps_ms = [ismissing(r.tp2_time) ? "À compl." : round(r.tp2_time*1000, digits=3) for r in resultats],
    TP3_Temps_ms = [ismissing(r.tp3_time) ? "NA" : round(r.tp3_time*1000, digits=3) for r in resultats],
)

println("\n📊 COMPARAISON TP2 vs TP3\n")
println(df_comparison)

## 9. Analyse et Conclusion

### ✓ TP3 (Programmation Dynamique) :
- **Coûts** : Résultats affichés ci-dessus
- **Performance** : À comparer avec TP2
- **Garanties** : Optimalité garantie ✓, Déterministe ✓

### ? TP2 (Branch-and-Bound) :
- À compléter après exécution du TP2

### 🎯 Comparaison :
1. Les résultats sont-ils **identiques** ?
2. Quelle approche est **plus rapide** ?
3. Quelle approche consomme **moins de mémoire** ?
4. Quelle est **plus facile à implémenter** ?

In [ ]:
# À compléter : Analyse personnelle

println("""
═══════════════════════════════════════════════════════════════════════════════
CONCLUSION GÉNÉRALE
═══════════════════════════════════════════════════════════════════════════════

À REMPLIR APRÈS COMPARAISON AVEC TP2 :

1. Résultats identiques ? ........... À compléter
2. Approche plus rapide ? ........... À compléter  
3. Approche recommandée ? .......... À compléter
4. Observations supplémentaires ? .. À compléter

═══════════════════════════════════════════════════════════════════════════════
""")